# Step 30 Map Schaefer200 To BOLD Run Grids

## What This Notebook Does

This notebook writes the standalone atlas-on-BOLD-grid files used to check whether the Schaefer parcel labels survive resampling onto each BOLD run grid.

## When To Run It

Run this notebook if you want to regenerate:
- atlas-preservation QC for Supplementary Table S4
- overlay images for atlas-on-grid provenance

You do **not** need Step 30 to run the main parcel export in Step 31. Step 31 remains the authoritative export path.

## Manuscript Linkage

- Main Methods 2.3
- Supplementary Methods 1.4
- Supplementary Results 2.4
- Supplementary Table S4 support
- Supplementary Figure S1C support

## Inputs Expected

- a derivatives root containing the fMRIPrep-style `.../func/` folders searched by this notebook
- matching brain masks when available
- a Schaefer2018 atlas NIfTI and matching TSV in template space
- optional raw BIDS JSON sidecars if you want extra provenance fields in the summary CSV

## Outputs Written

- one atlas-on-BOLD-grid NIfTI per run under `bold_parcel/atlas_on_boldgrid/`
- optional atlas overlay PNGs
- `qc_atlas_on_boldgrid_summary.csv`

In [ ]:
# Step 0. Import Python modules and locate the Stage-3 helper module

import sys
from pathlib import Path

STAGE3_DIR = Path.cwd()
candidate = Path.cwd() / "notebooks" / "3_bold"
if not (STAGE3_DIR / "stage3_bold_export_helpers.py").exists() and candidate.exists():
    STAGE3_DIR = candidate

if not (STAGE3_DIR / "stage3_bold_export_helpers.py").exists():
    raise FileNotFoundError(
        f"Stage-3 helper not found: {STAGE3_DIR / 'stage3_bold_export_helpers.py'}"
    )

if str(STAGE3_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE3_DIR.resolve()))

from stage3_bold_export_helpers import run_atlas_on_boldgrid

In [ ]:
# Step 1. User-editable roots and atlas settings
#
# FMRIPREP_ROOT should be the folder where this notebook can find the
# per-run `.../func/*boldref.nii.gz` files.
# DERIVATIVES_ROOT is where the Stage-3 atlas-on-grid outputs will be written.
# TEMPLATEFLOW_ROOT should contain `tpl-MNI152NLin2009cAsym/`.

FMRIPREP_ROOT = Path("<SET_FMRIPREP_ROOT>")
DERIVATIVES_ROOT = Path("<SET_DERIVATIVES_ROOT>")
TEMPLATEFLOW_ROOT = Path("<SET_TEMPLATEFLOW_ROOT>")

SPACE_TAG = "space-MNI152NLin2009cAsym"
ATLAS_FILENAME = "tpl-MNI152NLin2009cAsym_res-01_atlas-Schaefer2018_desc-200Parcels7Networks_dseg.nii.gz"

OVERWRITE = True
MAKE_OVERLAY_PNGS = True
RAW_BIDS_ROOT = None

In [ ]:
# Step 2. Build the derived paths and print a short run summary

FMRI_ROOT = FMRIPREP_ROOT
OUT_ROOT = DERIVATIVES_ROOT / "bold_parcel" / "atlas_on_boldgrid"
ATLAS_NII = TEMPLATEFLOW_ROOT / "tpl-MNI152NLin2009cAsym" / ATLAS_FILENAME
ATLAS_TSV = Path(str(ATLAS_NII).replace("_dseg.nii.gz", "_dseg.tsv"))


def assert_configured_path(path_value, label):
    path_str = str(path_value)
    if "<SET_" in path_str:
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")


assert_configured_path(FMRIPREP_ROOT, "FMRIPREP_ROOT")
assert_configured_path(DERIVATIVES_ROOT, "DERIVATIVES_ROOT")
assert_configured_path(TEMPLATEFLOW_ROOT, "TEMPLATEFLOW_ROOT")
if RAW_BIDS_ROOT is not None:
    assert_configured_path(RAW_BIDS_ROOT, "RAW_BIDS_ROOT")

print("Stage 3 / Step 30: atlas-on-grid preservation QC")
print(f"  fMRIPrep search root:  {FMRI_ROOT}")
print(f"  Output root:           {OUT_ROOT}")
print(f"  Atlas NIfTI:           {ATLAS_NII}")
print(f"  Atlas TSV:             {ATLAS_TSV}")
print(f"  Overwrite outputs:     {OVERWRITE}")
print(f"  Write overlay PNGs:    {MAKE_OVERLAY_PNGS}")
print("  Downstream use:        Step 32 joins this QC summary with the Step-31 export index for Table S4.")

In [ ]:
# Step 3. Run the atlas-preservation QC helper and review the summary table

qc_df = run_atlas_on_boldgrid(
    fmri_root=FMRI_ROOT,
    out_root=OUT_ROOT,
    atlas_nii=ATLAS_NII,
    atlas_tsv=ATLAS_TSV,
    space_tag=SPACE_TAG,
    overwrite=OVERWRITE,
    raw_bids_root=RAW_BIDS_ROOT,
    make_overlay_pngs=MAKE_OVERLAY_PNGS,
)

qc_df.head()